**Preproccessing data**

In [1]:
import pandas as pd
import numpy as np

# 1. Load Data
df = pd.read_csv('StudentPerformanceFactors (1).csv')

# 2. Seleksi Fitur yang diminta
features = ['Hours_Studied', 'Parental_Involvement', 'Access_to_Resources',
            'Sleep_Hours', 'Previous_Scores', 'Motivation_Level',
            'Tutoring_Sessions', 'Learning_Disabilities', 'Exam_Score']
df_model = df[features].copy()

# 3. Transformasi Data (Binning)
# Mengubah angka kontinu menjadi kategori agar bisa digunakan dalam aturan (Rules)
def bin_hours(x): return 'Low' if x < 15 else ('Medium' if x <= 25 else 'High')
def bin_scores(x): return 'Low' if x < 65 else ('Medium' if x <= 85 else 'High')
def bin_sleep(x): return 'Poor' if x < 6 else ('Normal' if x <= 8 else 'Good')
def bin_risk(x): return 'High Risk' if x < 65 else ('Moderate Risk' if x <= 70 else 'Low Risk')

df_model['Hours_Cat'] = df_model['Hours_Studied'].apply(bin_hours)
df_model['Prev_Score_Cat'] = df_model['Previous_Scores'].apply(bin_scores)
df_model['Sleep_Cat'] = df_model['Sleep_Hours'].apply(bin_sleep)
df_model['Academic_Risk'] = df_model['Exam_Score'].apply(bin_risk)

# Menampilkan hasil preprocessing
print("Dataset Siap Digunakan:")
display(df_model[['Hours_Cat', 'Prev_Score_Cat', 'Sleep_Cat', 'Academic_Risk']].head())

Dataset Siap Digunakan:


,Hours_Cat,Prev_Score_Cat,Sleep_Cat,Academic_Risk
0,Medium,Medium,Normal,Moderate Risk
1,Medium,Low,Normal,High Risk
2,Medium,High,Normal,Low Risk
3,High,High,Normal,Low Risk
4,Medium,Medium,Normal,Moderate Risk


melihat beberapa baris pertama untuk memastikan kolom baru (hasil binning/transformasi) sudah muncul.

In [5]:
# Menampilkan 5 data teratas pada kolom yang sudah diproses
print("Cek Sampel Data:")
display(df_model[['Hours_Cat', 'Prev_Score_Cat', 'Sleep_Cat', 'Academic_Risk']].head())

Cek Sampel Data:


,Hours_Cat,Prev_Score_Cat,Sleep_Cat,Academic_Risk
0,Medium,Medium,Normal,Moderate Risk
1,Medium,Low,Normal,High Risk
2,Medium,High,Normal,Low Risk
3,High,High,Normal,Low Risk
4,Medium,Medium,Normal,Moderate Risk


memastikan tidak ada nilai null dan tipe datanya sudah sesuai (biasanya menjadi object atau category setelah binning).

In [6]:
print("Informasi Dataset:")
df_model[['Hours_Cat', 'Prev_Score_Cat', 'Sleep_Cat', 'Academic_Risk']].info()

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Hours_Cat       6607 non-null   object
 1   Prev_Score_Cat  6607 non-null   object
 2   Sleep_Cat       6607 non-null   object
 3   Academic_Risk   6607 non-null   object
dtypes: object(4)
memory usage: 206.6+ KB


**Certainty Factor**

Menggunakan tingkat keyakinan pakar ($MB$ - Measure of Belief dan $MD$ - Measure of Disbelief).

In [4]:
# =================================================================
# METODE D: CERTAINTY FACTOR (CF)
# =================================================================

def calculate_cf_academic_risk(row):
    """
    Menghitung tingkat keyakinan (Certainty Factor) bahwa seorang siswa
    memiliki 'High Academic Risk' berdasarkan fakta yang ada.
    """

    # Nilai CF Pakar (Contoh Justifikasi Akademis: Jam belajar rendah
    # berkorelasi kuat dengan risiko kegagalan)
    # CF[H, E] = MB - MD (Dalam contoh ini kita asumsikan MD=0 untuk penyederhanaan)
    rules_cf = {
        'Hours_Cat_Low': 0.8,       # Sangat yakin jam belajar rendah memicu risiko
        'Prev_Score_Cat_Low': 0.7,  # Yakin nilai sebelumnya rendah memicu risiko
        'Sleep_Cat_Poor': 0.5,      # Cukup yakin kurang tidur memicu risiko
        'Motivation_Low': 0.4       # Kurang motivasi berdampak pada risiko
    }

    active_cf = []

    # 1. Cek Fakta dari Data
    if row['Hours_Cat'] == 'Low': active_cf.append(rules_cf['Hours_Cat_Low'])
    if row['Prev_Score_Cat'] == 'Low': active_cf.append(rules_cf['Prev_Score_Cat_Low'])
    if row['Sleep_Cat'] == 'Poor': active_cf.append(rules_cf['Sleep_Cat_Poor'])
    if row['Motivation_Level'] == 'Low': active_cf.append(rules_cf['Motivation_Low'])

    # 2. Kombinasi Certainty Factor (Metode Serial/Paralel)
    # Rumus: CF_gabungan = CF1 + CF2 * (1 - CF1)
    cf_final = 0
    if not active_cf:
        return 0

    cf_final = active_cf[0]
    for i in range(1, len(active_cf)):
        cf_final = cf_final + active_cf[i] * (1 - cf_final)

    return cf_final

# --- PENGUJIAN DENGAN DATASET ---
# Mengambil 5 sampel dari dataset hasil preprocessing
sample_data = df_model.head(5).copy()
sample_data['CF_High_Risk'] = sample_data.apply(calculate_cf_academic_risk, axis=1)

print("Hasil Analisis Certainty Factor untuk High Risk:")
display(sample_data[['Hours_Cat', 'Prev_Score_Cat', 'Sleep_Cat', 'CF_High_Risk']])

# --- PENGUJIAN DENGAN DATA MANUAL (Input User) ---
user_input = {
    'Hours_Cat': 'Low',
    'Prev_Score_Cat': 'Low',
    'Sleep_Cat': 'Poor',
    'Motivation_Level': 'Low'
}
final_score = calculate_cf_academic_risk(user_input)
print(f"\nKeyakinan Sistem untuk Input Manual: {final_score:.2%} (High Risk)")

Hasil Analisis Certainty Factor untuk High Risk:


,Hours_Cat,Prev_Score_Cat,Sleep_Cat,CF_High_Risk
0,Medium,Medium,Normal,0.40
1,Medium,Low,Normal,0.82
2,Medium,High,Normal,0.00
3,High,High,Normal,0.00
4,Medium,Medium,Normal,0.00



Keyakinan Sistem untuk Input Manual: 98.20% (High Risk)
